# Notebook 02: Error Metrics and Model Selection

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import curve_fit
from utils_isotherms import (
    langmuir, double_langmuir, freundlich, sips, toth, jovanovic,
    temkin, unilan, redlich_peterson, koble_corrigan, radke_prausnitz,
    dubinin_ra, hill, MODELS,
    all_error_metrics,
    compute_SSE, compute_AICc, compute_BIC, akaike_weights, f_test,
    extract_descriptors
)

## Load experimental data

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                        DATA FILE CONFIGURATION                              ║
# ║                                                                              ║
# ║  Set DATA_FILE below to the name of your experimental isotherm data file.   ║
# ║  Place the file in the SAME FOLDER as this notebook.                        ║
# ║                                                                              ║
# ║  File requirements:                                                          ║
# ║    - Format  : CSV (.csv) or Excel (.xlsx / .xls)                           ║
# ║    - Columns : Ce  (equilibrium concentration, mg/L)                        ║
# ║                qe  (adsorption capacity, mg/g)                              ║
# ║    - Column names are case-insensitive (Ce, ce, CE all work).               ║
# ║                                                                              ║
# ║  Examples:                                                                   ║
# ║    DATA_FILE = "sample_isotherm_data.csv"          # included sample        ║
# ║    DATA_FILE = "my_experiment.xlsx"                 # your own data         ║
# ║                                                                              ║
# ║  A sample file (sample_isotherm_data.csv) is included for testing.          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

DATA_FILE = "sample_isotherm_data.csv"   # <--- CHANGE THIS to your data file

# ══════════════════════════════════════════════════════════════════════════════
#  Do NOT modify anything below this line unless you know what you are doing.
# ══════════════════════════════════════════════════════════════════════════════

# ── Resolve path relative to this notebook's folder ───────────────────────────
_NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
DATA_PATH = os.path.join(_NOTEBOOK_DIR, DATA_FILE)

# ── Validate input ────────────────────────────────────────────────────────────
if not DATA_FILE or DATA_FILE.strip() == "":
    raise ValueError(
        "\n\n"
        "  *** ERROR: No data file specified! ***\n\n"
        "  Go to the top of this cell and set DATA_FILE to your file name.\n\n"
        "  Example:\n"
        "      DATA_FILE = 'sample_isotherm_data.csv'\n"
    )

if not os.path.isfile(DATA_PATH):
    raise FileNotFoundError(
        f"\n\n"
        f"  *** ERROR: File not found: '{DATA_FILE}' ***\n\n"
        f"  Looked in: {_NOTEBOOK_DIR}\n\n"
        f"  Place your data file in the same folder as this notebook.\n"
    )

# ── Load data ─────────────────────────────────────────────────────────────────
ext = os.path.splitext(DATA_FILE)[1].lower()
if ext == '.csv':
    df_data = pd.read_csv(DATA_PATH)
elif ext in ('.xlsx', '.xls'):
    df_data = pd.read_excel(DATA_PATH)
else:
    raise ValueError(
        f"\n  Unsupported file format '{ext}'. Use .csv or .xlsx\n"
    )

# Normalise column names (case-insensitive)
col_map = {c.strip().lower(): c for c in df_data.columns}
if 'ce' not in col_map or 'qe' not in col_map:
    raise ValueError(
        f"\n\n"
        f"  *** ERROR: Required columns not found! ***\n\n"
        f"  Your file has columns: {list(df_data.columns)}\n"
        f"  This notebook expects two columns named 'Ce' and 'qe'\n"
        f"  (case-insensitive: Ce, ce, CE, etc. all work).\n"
    )

df_data = df_data.rename(columns={col_map['ce']: 'Ce', col_map['qe']: 'qe'})
df_data = df_data[['Ce', 'qe']].dropna()
df_data = df_data[(df_data['Ce'] > 0) & (df_data['qe'] >= 0)].sort_values('Ce')

if len(df_data) < 4:
    raise ValueError(
        f"\n\n"
        f"  *** ERROR: Not enough valid data points ({len(df_data)})! ***\n\n"
        f"  At least 4 data points are needed for meaningful fitting.\n"
        f"  Check your file for missing or negative values.\n"
    )

C_exp = df_data['Ce'].to_numpy(dtype=float)
q_exp = df_data['qe'].to_numpy(dtype=float)
n_points = len(q_exp)

print(f"Loaded {n_points} data points from '{DATA_FILE}'")
print(f"  Ce range : {C_exp.min():.4g} -- {C_exp.max():.4g} mg/L")
print(f"  qe range : {q_exp.min():.4g} -- {q_exp.max():.4g} mg/g")

# ── Data-adaptive scaling for initial guesses and bounds ──────────────────────
_Qg  = np.max(q_exp) * 1.5           # initial Qmax guess (50% above observed max)
_Kg  = 1.0 / max(np.median(C_exp), 1e-8)  # affinity guess ~ 1/median(Ce)
_Qhi = np.max(q_exp) * 20            # upper bound for Qmax-like parameters

# ── Fitting configuration: {model_name: (function, p0, bounds_lo, bounds_hi)} ──
fit_config = {
    'Langmuir':         (langmuir,          [_Qg, _Kg],
                         [0, 0],             [_Qhi, 100]),
    'Double Langmuir':  (double_langmuir,   [_Qg*0.5, _Kg*2, _Qg*0.5, _Kg*0.5],
                         [0, 0, 0, 0],       [_Qhi, 100, _Qhi, 100]),
    'Freundlich':       (freundlich,         [np.max(q_exp)/np.max(C_exp)**0.5, 0.4],
                         [0, 0.01],          [_Qhi*10, 1]),
    'Sips':             (sips,              [_Qg, _Kg, 0.8],
                         [0, 0, 0.1],        [_Qhi, 100, 2]),
    'Toth':             (toth,              [_Qg, _Kg, 0.8],
                         [0, 0, 0.1],        [_Qhi, 100, 2]),
    'Jovanovic':        (jovanovic,         [_Qg, _Kg],
                         [0, 0],             [_Qhi, 100]),
    'Temkin':           (temkin,            [1.0, 0.1],
                         [1e-6, 1e-6],       [1e6, 100]),
    'UNILAN':           (unilan,            [_Qg, _Kg*5, _Kg*0.1],
                         [0, 1e-6, 1e-8],    [_Qhi, 1000, 100]),
    'Redlich-Peterson': (redlich_peterson,  [_Qg*_Kg, _Kg, 0.8],
                         [0, 0, 0.01],       [1e4, 1000, 1]),
    'Koble-Corrigan':   (koble_corrigan,    [_Qg*_Kg, _Kg, 0.4],
                         [0, 0, 0.01],       [_Qhi*10, 1000, 1]),
    'Radke-Prausnitz':  (radke_prausnitz,   [_Qg, _Qg*_Kg, 0.5],
                         [0, 0, 0],          [1e4, 1e4, 1]),
    'D-R/D-A':          (dubinin_ra,        [_Qg, 10, 2],
                         [0, 0.1, 1],        [_Qhi, 100, 10]),
    'Hill':             (hill,              [_Qg, _Qg/_Kg, 0.8],
                         [0, 1e-6, 0.1],     [_Qhi, 1e6, 5]),
}

# ── Fit all models ────────────────────────────────────────────────────────────
results = {}  # model_name -> {popt, q_pred, n_params, success}
for name, (func, p0, blo, bhi) in fit_config.items():
    try:
        popt, _ = curve_fit(func, C_exp, q_exp, p0=p0,
                            bounds=(blo, bhi), maxfev=20000)
        q_pred = func(C_exp, *popt)
        results[name] = {'popt': popt, 'q_pred': q_pred,
                         'n_params': len(popt), 'success': True}
    except Exception as e:
        print(f"  {name}: FAILED -- {e}")
        results[name] = {'popt': None, 'q_pred': None,
                         'n_params': len(p0), 'success': False}

# Print fitted parameters
print(f"\n{'Model':<20s} {'Params':>6s}  Fitted values")
print("-" * 70)
for name, res in results.items():
    if res['success']:
        param_str = ', '.join(f'{v:.4f}' for v in res['popt'])
        print(f"{name:<20s} {res['n_params']:>6d}  {param_str}")
    else:
        print(f"{name:<20s}  -- fit failed --")

print(f"\nSuccessful fits: {sum(r['success'] for r in results.values())}/{len(results)}")

## Compute all error metrics

In [ ]:
# Compute error metrics for all successfully fitted models
metrics_dict = {}
for name, res in results.items():
    if res['success']:
        metrics_dict[name] = all_error_metrics(q_exp, res['q_pred'], res['n_params'])

df_metrics = pd.DataFrame(metrics_dict)

print("Error Metrics Comparison (all models):")
print(df_metrics.round(4).to_string())

## Information criteria and Akaike weights

In [ ]:
# Compute IC for all successfully fitted models
ic_rows = []
for name, res in results.items():
    if res['success']:
        sse = compute_SSE(q_exp, res['q_pred'])
        aicc = compute_AICc(n_points, sse, res['n_params'])
        bic = compute_BIC(n_points, sse, res['n_params'])
        ic_rows.append({'Model': name, 'SSE': sse, 'n_params': res['n_params'],
                        'AICc': aicc, 'BIC': bic})

df_ic = pd.DataFrame(ic_rows)
df_ic['delta_AICc'] = df_ic['AICc'] - df_ic['AICc'].min()
df_ic['Akaike_weight'] = akaike_weights(df_ic['AICc'].values)
df_ic = df_ic.sort_values('AICc').reset_index(drop=True)

print("Information Criteria and Model Selection (sorted by AICc):")
print(df_ic.round(4).to_string(index=False))
print(f"\nBest model by AICc: {df_ic.iloc[0]['Model']}")
print("\nInterpretation: delta_AICc < 2 -> substantial support | 4-7 -> less support | > 10 -> no support")

## F-test for nested model pairs

In [ ]:
# F-test for nested model pairs (alpha = 0.05)
# Nested relationships: simpler model reduces to special case of the complex one
nested_pairs = [
    ('Langmuir',    'Sips',             'Sips m->1 reduces to Langmuir'),
    ('Langmuir',    'Toth',             'Toth t->1 reduces to Langmuir'),
    ('Langmuir',    'Redlich-Peterson', 'R-P beta->1 reduces to Langmuir'),
    ('Langmuir',    'Hill',             'Hill nH->1 reduces to Langmuir'),
    ('Langmuir',    'Double Langmuir',  'DL K1=K2, Q2=0 reduces to Langmuir'),
    ('Freundlich',  'Koble-Corrigan',   'KC B->0 reduces to Freundlich'),
    ('Jovanovic',   'D-R/D-A',          'D-A generalises Jovanovic-like shape'),
]

print("F-tests for Nested Model Pairs (alpha = 0.05)")
print("=" * 80)
for simple, full, note in nested_pairs:
    r_s, r_f = results.get(simple, {}), results.get(full, {})
    if r_s.get('success') and r_f.get('success'):
        sse_s = compute_SSE(q_exp, r_s['q_pred'])
        sse_f = compute_SSE(q_exp, r_f['q_pred'])
        p_s, p_f = r_s['n_params'], r_f['n_params']
        if p_f > p_s:
            F_stat, F_crit, pval, reject = f_test(sse_s, p_s, sse_f, p_f, n_points)
            tag = "REJECT simpler" if reject else "KEEP simpler"
            print(f"\n{simple} ({p_s}p) vs {full} ({p_f}p)  --  {note}")
            print(f"  F = {F_stat:.4f},  F_crit = {F_crit:.4f},  p = {pval:.6f}  ->  {tag}")
    else:
        which = []
        if not r_s.get('success'): which.append(simple)
        if not r_f.get('success'): which.append(full)
        print(f"\n{simple} vs {full}: skipped (fit failed for {', '.join(which)})")

## Visualization 1: Fit comparison and residuals

In [ ]:
# Shared colour palette for all models
import matplotlib.cm as cm

fitted_names = [n for n in results if results[n]['success']]
_tab = cm.tab20(np.linspace(0, 1, 20))
model_cmap = {name: _tab[i % 20] for i, name in enumerate(fitted_names)}

# Line styles cycle
_ls = ['-', '--', '-.', ':']

# Fine grid for smooth curves
Ce_fine = np.linspace(C_exp.min() * 0.5, C_exp.max() * 1.1, 500)

fig, axes = plt.subplots(2, 1, figsize=(12, 9),
                         gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# --- Top panel: all model fits ---
ax = axes[0]
ax.scatter(C_exp, q_exp, s=60, c='k', zorder=5, label='Experimental')
for i, name in enumerate(fitted_names):
    func = fit_config[name][0]
    popt = results[name]['popt']
    q_fine = func(Ce_fine, *popt)
    ax.plot(Ce_fine, q_fine, ls=_ls[i % 4], color=model_cmap[name],
            lw=2, label=name)
ax.set_ylabel('$q_e$ (mg g$^{-1}$)', fontsize=12)
ax.legend(fontsize=8, ncol=3, loc='lower right')
ax.set_title(f'Isotherm Fits — All Models ({os.path.basename(DATA_FILE)})', fontsize=13)

# --- Bottom panel: residuals (top 5 by AICc) ---
ax2 = axes[1]
top5 = df_ic['Model'].head(5).tolist()
offsets = np.linspace(-0.8, 0.8, len(top5))
for j, name in enumerate(top5):
    res_vals = q_exp - results[name]['q_pred']
    ax2.stem(C_exp + offsets[j], res_vals, linefmt='-', markerfmt='o',
             basefmt='k-', label=name)
    col = model_cmap[name]
    plt.setp(ax2.containers[-1].markerline, color=col, markersize=4)
    plt.setp(ax2.containers[-1].stemlines, color=col, alpha=0.6)
ax2.axhline(0, color='k', lw=0.8)
ax2.set_xlabel('$C_e$ (mg L$^{-1}$)', fontsize=12)
ax2.set_ylabel('Residual', fontsize=12)
ax2.legend(fontsize=8, ncol=5)

plt.tight_layout()
plt.show()

## Visualization 2: Residual diagnostics (per model)

In [ ]:
# Residual diagnostics for top 4 models (by AICc)
top4 = df_ic['Model'].head(4).tolist()

fig, axes = plt.subplots(len(top4), 3, figsize=(14, 3 * len(top4)))

for i, name in enumerate(top4):
    res_vals = q_exp - results[name]['q_pred']
    col = model_cmap[name]

    # Column 1: residuals vs Ce
    ax = axes[i, 0]
    ax.scatter(C_exp, res_vals, c=[col], edgecolors='k', s=40, zorder=3)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_ylabel(f'{name}\nResidual', fontsize=10)
    if i == len(top4) - 1:
        ax.set_xlabel('$C_e$ (mg L$^{-1}$)', fontsize=10)
    if i == 0:
        ax.set_title('Residuals vs $C_e$', fontsize=11)

    # Column 2: residuals vs predicted
    ax = axes[i, 1]
    ax.scatter(results[name]['q_pred'], res_vals, c=[col], edgecolors='k', s=40, zorder=3)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    if i == len(top4) - 1:
        ax.set_xlabel('$q_{pred}$ (mg g$^{-1}$)', fontsize=10)
    if i == 0:
        ax.set_title('Residuals vs $q_{pred}$', fontsize=11)

    # Column 3: histogram of residuals
    ax = axes[i, 2]
    ax.hist(res_vals, bins=10, color=col, alpha=0.7, edgecolor='k')
    ax.axvline(0, color='k', lw=0.8, ls='--')
    if i == len(top4) - 1:
        ax.set_xlabel('Residual', fontsize=10)
    if i == 0:
        ax.set_title('Residual distribution', fontsize=11)

plt.suptitle('Residual Diagnostics (Top 4 Models by AICc)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Visualization 3: Error metrics comparison (grouped bar chart)

In [ ]:
# Error metrics comparison — all models (grouped bar chart)
abs_metrics = ['SSE', 'EABS', 'HYBRID']
rel_metrics = ['ARED', 'MPSED', 'RESID']
qual_metrics = ['R2', 'R2adj']

fitted_cols = [n for n in df_metrics.columns]
n_models = len(fitted_cols)
bar_colors = [model_cmap.get(m, 'grey') for m in fitted_cols]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel (a): absolute error metrics (log scale)
ax = axes[0]
x_pos = np.arange(len(abs_metrics))
width = 0.8 / n_models
for j, model in enumerate(fitted_cols):
    vals = [df_metrics.loc[m, model] for m in abs_metrics]
    ax.bar(x_pos + j * width, vals, width, color=bar_colors[j],
           edgecolor='k', linewidth=0.5, label=model)
ax.set_xticks(x_pos + width * n_models / 2)
ax.set_xticklabels(abs_metrics, fontsize=11)
ax.set_yscale('log')
ax.set_ylabel('Value (log scale)', fontsize=11)
ax.set_title('(a) Absolute error metrics', fontsize=12)

# Panel (b): relative error metrics
ax = axes[1]
x_pos = np.arange(len(rel_metrics))
for j, model in enumerate(fitted_cols):
    vals = [df_metrics.loc[m, model] for m in rel_metrics]
    ax.bar(x_pos + j * width, vals, width, color=bar_colors[j],
           edgecolor='k', linewidth=0.5, label=model)
ax.set_xticks(x_pos + width * n_models / 2)
ax.set_xticklabels(rel_metrics, fontsize=11)
ax.set_ylabel('Value', fontsize=11)
ax.set_title('(b) Relative error metrics', fontsize=12)

# Panel (c): R2 and R2adj
ax = axes[2]
x_pos = np.arange(len(qual_metrics))
for j, model in enumerate(fitted_cols):
    vals = [df_metrics.loc[m, model] for m in qual_metrics]
    ax.bar(x_pos + j * width, vals, width, color=bar_colors[j],
           edgecolor='k', linewidth=0.5, label=model)
ax.set_xticks(x_pos + width * n_models / 2)
ax.set_xticklabels(['$R^2$', '$R^2_{adj}$'], fontsize=11)
ax.set_ylabel('Value', fontsize=11)
ax.set_title('(c) Goodness of fit', fontsize=12)

# Single legend below
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, fontsize=8,
           bbox_to_anchor=(0.5, -0.08))
plt.suptitle('Error Metrics Comparison Across All Models', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Visualization 4: Information criteria and Akaike weights

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Use df_ic which is sorted by AICc
ic_names = df_ic['Model'].tolist()

# Panel (a): AICc and BIC side by side
ax = axes[0]
x_pos = np.arange(len(ic_names))
width = 0.35
ax.bar(x_pos - width/2, df_ic['AICc'], width, color='steelblue',
       edgecolor='k', label='AICc')
ax.bar(x_pos + width/2, df_ic['BIC'], width, color='coral',
       edgecolor='k', label='BIC')
ax.set_xticks(x_pos)
ax.set_xticklabels(ic_names, fontsize=8, rotation=45, ha='right')
ax.set_ylabel('Criterion value', fontsize=11)
ax.set_title('(a) AICc and BIC', fontsize=12)
ax.legend(fontsize=10)

# Panel (b): delta AICc with threshold bands
ax = axes[1]
delta_vals = df_ic['delta_AICc'].values
bar_cols = ['#2ca02c' if d < 2 else '#ff7f0e' if d < 10 else '#d62728'
            for d in delta_vals]
bars = ax.bar(range(len(ic_names)), delta_vals, color=bar_cols, edgecolor='k')
ax.set_xticks(range(len(ic_names)))
ax.set_xticklabels(ic_names, fontsize=8, rotation=45, ha='right')
ax.axhline(2, color='green', ls='--', lw=1.2, label='$\\Delta$AICc = 2')
ax.axhline(10, color='red', ls='--', lw=1.2, label='$\\Delta$AICc = 10')
ax.axhspan(0, 2, alpha=0.08, color='green')
ax.axhspan(2, 10, alpha=0.08, color='orange')
if max(delta_vals) > 10:
    ax.axhspan(10, max(delta_vals) * 1.1, alpha=0.08, color='red')
ax.set_ylabel('$\\Delta$AICc', fontsize=11)
ax.set_title('(b) $\\Delta$AICc thresholds', fontsize=12)
ax.legend(fontsize=9)

# Panel (c): Akaike weights (horizontal bar — scales better than pie for 13 models)
ax = axes[2]
w_vals = df_ic['Akaike_weight'].values
ax.barh(ic_names[::-1], w_vals[::-1],
        color=[model_cmap.get(m, 'grey') for m in ic_names[::-1]], edgecolor='k')
ax.set_xlabel('Akaike weight', fontsize=11)
ax.set_title('(c) Akaike weights', fontsize=12)
for j, (v, n) in enumerate(zip(w_vals[::-1], ic_names[::-1])):
    if v > 0.01:
        ax.text(v + 0.005, j, f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('Model Selection by Information Criteria', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Visualization 5: Radar chart — normalised error metrics

In [ ]:
# Radar chart — top 6 models by AICc
radar_metrics = ['SSE', 'EABS', 'RESID', 'ARED', 'MPSED', 'HYBRID']
top6 = df_ic['Model'].head(6).tolist()

df_radar = df_metrics.loc[radar_metrics, top6].copy()

# Min-max normalise per row, inverted: 1 = best, 0 = worst
df_norm = df_radar.apply(
    lambda row: 1 - (row - row.min()) / (row.max() - row.min() + 1e-15), axis=1)

labels = radar_metrics
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for name in top6:
    values = df_norm[name].tolist() + [df_norm[name].tolist()[0]]
    ax.plot(angles, values, 'o-', lw=2, color=model_cmap[name], label=name)
    ax.fill(angles, values, alpha=0.08, color=model_cmap[name])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8, color='grey')
ax.set_title('Normalised Error Metrics — Top 6 Models\n(1 = best, 0 = worst)', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)

plt.tight_layout()
plt.show()

## Visualization 6: Parity plot (predicted vs experimental)

In [ ]:
# Parity plots — top 6 models by AICc
top6 = df_ic['Model'].head(6).tolist()
nrows, ncols = 2, 3

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 10))
axes_flat = axes.flatten()

for i, name in enumerate(top6):
    ax = axes_flat[i]
    q_pred = results[name]['q_pred']
    r2 = df_metrics.loc['R2', name]
    ax.scatter(q_exp, q_pred, s=50, c=[model_cmap[name]],
               edgecolors='k', zorder=3)
    lims = [min(q_exp.min(), q_pred.min()) * 0.9,
            max(q_exp.max(), q_pred.max()) * 1.1]
    ax.plot(lims, lims, 'k--', lw=1, label='1:1 line')
    ax.fill_between(lims, [l * 0.9 for l in lims], [l * 1.1 for l in lims],
                    alpha=0.08, color='grey', label='$\\pm$10%')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_aspect('equal')
    ax.set_xlabel('$q_{exp}$ (mg g$^{-1}$)', fontsize=10)
    ax.set_ylabel('$q_{pred}$ (mg g$^{-1}$)', fontsize=10)
    ax.set_title(f'{name}  ($R^2$ = {r2:.4f})', fontsize=11)
    ax.legend(fontsize=8, loc='upper left')

plt.suptitle('Parity Plots — Top 6 Models by AICc', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Visualization 7: Heatmap — all metrics at a glance

In [ ]:
# Heatmap — all metrics, all models
all_rows = df_metrics.copy()
all_rows.loc['1-R2'] = 1 - all_rows.loc['R2']
all_rows.loc['1-R2adj'] = 1 - all_rows.loc['R2adj']
all_rows = all_rows.drop(['R2', 'R2adj'])

# Add IC rows from df_ic
for _, row in df_ic.iterrows():
    all_rows.loc['AICc', row['Model']] = row['AICc']
    all_rows.loc['BIC', row['Model']] = row['BIC']

# Normalise per row, inverted: 1 = best, 0 = worst
df_heat = all_rows.apply(
    lambda row: 1 - (row - row.min()) / (row.max() - row.min() + 1e-15), axis=1)

fig, ax = plt.subplots(figsize=(max(10, len(fitted_names) * 0.9), 7))
im = ax.imshow(df_heat.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(df_heat.shape[1]))
ax.set_xticklabels(df_heat.columns, fontsize=9, rotation=45, ha='right')
ax.set_yticks(range(df_heat.shape[0]))
ax.set_yticklabels(df_heat.index, fontsize=10)

for i in range(df_heat.shape[0]):
    for j in range(df_heat.shape[1]):
        val = all_rows.iloc[i, j]
        if np.isnan(val):
            txt = '—'
        elif abs(val) < 100:
            txt = f'{val:.2f}'
        else:
            txt = f'{val:.1f}'
        colour = 'white' if df_heat.iloc[i, j] < 0.4 else 'black'
        ax.text(j, i, txt, ha='center', va='center', fontsize=7, color=colour)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Normalised value (1 = best)', fontsize=10)
ax.set_title('Model Comparison Heatmap (green = best, red = worst)', fontsize=13)
plt.tight_layout()
plt.show()

## Visualization 8: Confidence bands from covariance matrix

In [ ]:
from scipy.stats import t as t_dist

# Confidence bands for Langmuir and Sips (Monte Carlo from covariance)
Ce_fine_cb = np.linspace(0.01, 110, 500)
n_mc = 500
rng = np.random.default_rng(0)

cb_models = [
    ('Langmuir', langmuir, [100, 0.1], ([0, 0], [500, 10])),
    ('Sips',     sips,     [100, 0.08, 0.8], ([0, 0, 0.1], [500, 10, 2])),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (name, func, p0, (blo, bhi)) in zip(axes, cb_models):
    popt_cb, pcov_cb = curve_fit(func, C_exp, q_exp, p0=p0,
                                  bounds=(blo, bhi), maxfev=10000)
    col = model_cmap.get(name, 'steelblue')

    q_mc = np.zeros((n_mc, len(Ce_fine_cb)))
    for k in range(n_mc):
        p_sample = rng.multivariate_normal(popt_cb, pcov_cb)
        p_sample = np.clip(p_sample, 1e-6, None)
        try:
            q_mc[k] = func(Ce_fine_cb, *p_sample)
        except Exception:
            q_mc[k] = np.nan

    q_lo = np.nanpercentile(q_mc, 2.5, axis=0)
    q_hi = np.nanpercentile(q_mc, 97.5, axis=0)
    q_med = func(Ce_fine_cb, *popt_cb)

    ax.scatter(C_exp, q_exp, s=40, c='k', zorder=5, label='Data')
    ax.plot(Ce_fine_cb, q_med, '-', color=col, lw=2, label='Best fit')
    ax.fill_between(Ce_fine_cb, q_lo, q_hi, color=col, alpha=0.2,
                    label='95% confidence band')
    ax.set_xlabel('$C_e$ (mg L$^{-1}$)', fontsize=11)
    ax.set_ylabel('$q_e$ (mg g$^{-1}$)', fontsize=11)
    ax.set_title(f'{name} — 95% Confidence Band', fontsize=12)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Universal descriptor extraction — ln K*_aff for each model

In [ ]:
# Extract universal descriptors {Qmax, K*aff, ln K*aff, Eads, sigma_H} for every fitted model
param_names_map = {
    'Langmuir':         ['Qmax', 'KL'],
    'Double Langmuir':  ['Q1', 'K1', 'Q2', 'K2'],
    'Freundlich':       ['KF', '1/n'],
    'Sips':             ['Qmax', 'KS', 'm'],
    'Toth':             ['Qmax', 'b', 't'],
    'Jovanovic':        ['Qmax', 'b'],
    'Temkin':           ['aT', 'bT'],
    'UNILAN':           ['Qmax', 'bmax', 'bmin'],
    'Redlich-Peterson': ['KRP', 'aRP', 'beta'],
    'Koble-Corrigan':   ['A', 'B', '1/n'],
    'Radke-Prausnitz':  ['a', 'r', 'p'],
    'D-R/D-A':          ['Qmax', 'Ea', 'nDA'],
    'Hill':             ['Qmax', 'KD', 'nH'],
}

desc_rows = []
for name, res in results.items():
    if res['success']:
        pnames = param_names_map[name]
        params_dict = dict(zip(pnames, res['popt']))
        desc = extract_descriptors(name, params_dict)
        Kaff = desc['Kaff_star']
        ln_Kaff = np.log(Kaff) if (np.isfinite(Kaff) and Kaff > 0) else np.nan
        desc_rows.append({
            'Model': name,
            'Q_max (mg/g)': desc['Qmax'],
            'K*_aff': Kaff,
            'ln K*_aff': ln_Kaff,
            'E_ads (kJ/mol)': desc['Eads'],
            'sigma_H (kJ/mol)': desc['sigma_H'],
        })

df_desc = pd.DataFrame(desc_rows)

print("Universal Descriptors — All Models")
print("=" * 95)
print(df_desc.to_string(index=False, float_format='%.4f'))
print()
print("Key relations:")
print("  K*_aff = (K_H / Q_max) * C0   (dimensionless affinity constant)")
print("  ln K*_aff                       (natural log of K*_aff)")
print("  E_ads = RT * ln(K*_aff)         (standard-state adsorption energy, kJ/mol at 298 K)")
print("  sigma_H                         (energy distribution width, kJ/mol)")
print()
print("Notes:")
print("  - Freundlich: Q_max = NaN (no saturation), sigma_H = inf (unbounded distribution)")
print("  - Temkin, Radke-Prausnitz: sigma_H = NaN (no closed-form expression)")
print("  - Redlich-Peterson: sigma_H = NaN (empirical hybrid model)")